# R7: диагностика чистых отключений GNSS на validation

**Этот notebook подготовлен для ручного Save & Run All. Он не выполнялся при подготовке.**
Обучения, настройки параметров и test-оценки здесь нет. Используются сохранённые
R7 seed=0: full, отдельно обученная gps_only, FusionLSTM. Сравнение с Held-GNSS,
причинной экстраполяцией постоянной скорости и отдельным zero-IMU probing full checkpoint.
Transformer остаётся **некаузальной оконной моделью**, без добавления causal masks.

Добавьте сохранённый output notebook
[shapok/drone-nav-full-run, версия 7, output ID 352644733](https://www.kaggle.com/code/shapok/drone-nav-full-run?scriptVersionId=352644733)
как input этого нового notebook. Нужны его `drone/data/processed/` и `outputs/checkpoints/`.
Версия проверяется по SHA-256 всех десяти файлов; другой export будет отклонён.
Исходный сырой датасет добавлять не требуется. Код и конфигурация уже встроены:
GitHub, Интернет, API-ключи и установка зависимостей не нужны.

GPU можно включить; CPU также поддерживается. Нужны имеющиеся в Kaggle NumPy и PyTorch
с `torch.load(weights_only=True)`. Сохранённые R7 данные только читаются из `/kaggle/input`.
Запустите **Save Version → Save & Run All** в новом notebook. Для повторного полного запуска
используйте свежую сессию: существующий каталог результатов намеренно не перезаписывается.


In [ ]:
from pathlib import Path
import sys

CODE_ROOT = Path('/kaggle/working/diagnostic_clean_code')
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/outputs/diagnostic_clean_outages')
# Обычно оставьте None: поиск только среди добавленных inputs, затем строгая сверка хешей.
R7_ROOT = None


In [ ]:
import hashlib

PACKAGED_FILES = {'src/diagnostics/__init__.py': '"""Isolated, inference-only diagnostics; the R7 pipeline is unchanged."""\n', 'src/diagnostics/clean_outages.py': '"""Clean outages on the existing R7 coordinate/time representation.\n\nNo training, GT-dependent input construction, alignment, or filesystem writes.\nAvailability is a link-level flag, NOT a claim that a new fix arrived. The\nseparate update mask and source indices make that distinction auditable.\n"""\nimport hashlib\n\nimport numpy as np\n\n\ndef _timeline(t):\n    t = np.asarray(t, dtype=np.float64)\n    if t.ndim != 1 or not len(t) or not np.isfinite(t).all() or (np.diff(t) <= 0).any():\n        raise ValueError("timestamps must be finite, nonempty and strictly increasing")\n    return t\n\n\ndef false_runs(available):\n    """Half-open index intervals in the FINAL mask, independent of block order."""\n    missing = ~np.asarray(available, dtype=bool)\n    edges = np.diff(np.r_[False, missing, False].astype(np.int8))\n    return list(zip(np.flatnonzero(edges == 1).tolist(),\n                    np.flatnonzero(edges == -1).tolist()))\n\n\ndef availability_from_blocks(n, blocks):\n    available = np.ones(n, dtype=bool)\n    for start, stop in blocks:\n        if not (0 <= start < stop <= n):\n            raise ValueError("blocks must be nonempty half-open intervals within the record")\n        available[start:stop] = False\n    return available\n\n\ndef mask_sha256(available):\n    """Hash uint8 availability bytes, one byte per common-grid sample."""\n    return hashlib.sha256(np.asarray(available, dtype=np.uint8).tobytes()).hexdigest()\n\n\ndef common_validation_indices(t_imu, t_gt, train_fraction, val_fraction, window):\n    """Split first, restrict to original GT support, then trim incomplete windows.\n\n    All methods use these SAME indices. No GT extrapolation, no concatenation\n    across holes, no carry-over from train/test. Windows start at common[0].\n    """\n    t = _timeline(t_imu)\n    g = _timeline(t_gt)\n    if window < 1 or not (0 < train_fraction < 1 and 0 < val_fraction < 1 - train_fraction):\n        raise ValueError("invalid split/window")\n    start = int(len(t) * train_fraction)\n    stop = start + int(len(t) * val_fraction)\n    raw = np.arange(start, stop, dtype=np.int64)\n    supported = raw[(t[raw] >= g[0]) & (t[raw] <= g[-1])]\n    if len(supported) and (np.diff(supported) != 1).any():\n        raise ValueError("noncontiguous supported validation region")\n    n = len(supported) // window * window\n    if not n:\n        raise ValueError("validation has no complete supported window")\n    common = supported[:n]\n    return common, {\n        "split": "validation", "original_split_indices_half_open": [start, stop],\n        "original_split_n": len(raw), "gt_support_s": [float(g[0]), float(g[-1])],\n        "excluded_outside_gt_support_n": len(raw) - len(supported),\n        "discarded_incomplete_window_n": len(supported) - n,\n        "common_indices_half_open": [int(common[0]), int(common[-1]) + 1],\n        "common_n": n, "window": window, "stride": window, "window_count": n // window,\n        "common_first_timestamp_s": float(t[common[0]]),\n        "common_last_timestamp_s": float(t[common[-1]]),\n    }\n\n\ndef make_scenarios(t, protocol):\n    """Choose starts from timestamps only; never inspect positions/errors.\n\n    Three fractions of the admissible onset interval are snapped forward to a\n    grid point. Every duration uses the same starts, in separate full-trace\n    scenarios. [onset, cutoff) is unavailable. RNG is explicitly unused.\n    """\n    t = _timeline(t)\n    durations = protocol["durations_s"]\n    fractions = protocol["start_fractions"]\n    warmup, recovery = protocol["warmup_s"], protocol["recovery_s"]\n    if min(durations) <= 0 or warmup <= 0 or recovery <= 0:\n        raise ValueError("positive durations, warmup and recovery required")\n    if any(not 0 <= f <= 1 for f in fractions):\n        raise ValueError("start fractions must be in [0, 1]")\n    lo, hi = t[0] + warmup, t[-1] - recovery - max(durations)\n    if hi <= lo:\n        raise ValueError("validation too short for the fixed protocol")\n    starts = [int(np.searchsorted(t, lo + f * (hi - lo), side="left")) for f in fractions]\n    if len(set(starts)) != len(starts):\n        raise ValueError("not enough distinct starts; do not silently change protocol")\n    scenarios = [{"scenario_id": "val_control", "split": "validation",\n                  "episode_id": None, "requested_duration_s": 0,\n                  "source_blocked_intervals_s": [], "unavailable_index_runs": [],\n                  "n_unavailable": 0, "metric_scope": "whole_common_trace"}]\n    for episode, start in enumerate(starts, 1):\n        for duration in durations:\n            onset = float(t[start])\n            cutoff = onset + duration\n            stop = int(np.searchsorted(t, cutoff, side="left"))\n            if start == 0 or stop >= len(t) or t[-1] - t[stop] < recovery:\n                raise ValueError("episode lacks warmup/recovery; protocol cannot be used")\n            scenarios.append({\n                "scenario_id": f"val_e{episode}_d{duration:g}s", "split": "validation",\n                "episode_id": episode, "requested_duration_s": duration,\n                "target_onset_s": float(lo + fractions[episode - 1] * (hi - lo)),\n                "onset_s": onset, "requested_cutoff_s": cutoff,\n                "last_unavailable_s": float(t[stop - 1]),\n                "first_link_available_after_s": float(t[stop]),\n                "observed_mask_interval_s": float(t[stop] - t[start]),\n                "unavailable_sample_span_s": float(t[stop - 1] - t[start]),\n                "available_history_span_s": float(t[start] - t[0]),\n                "recovery_span_s": float(t[-1] - t[stop]),\n                "source_blocked_intervals_s": [[onset, cutoff]],\n                "unavailable_index_runs": [[start, stop]], "n_unavailable": stop - start,\n                "metric_scope": "outage_only",\n            })\n    for scenario in scenarios:\n        available = availability_from_blocks(len(t), scenario["unavailable_index_runs"])\n        scenario.update(n_common=len(t), mask_uint8_sha256=mask_sha256(available),\n                        generator_seed=protocol["generator_seed"], rng_used=False)\n    return scenarios\n\n\ndef clean_gnss(t, candidates, source_indices, source_timestamps, available,\n               blocked_intervals=None):\n    """Sequentially accept only legal candidate fixes after forming final mask.\n\n    Candidates are the unchanged R7 GPS resampling. Source timestamps/indices\n    refer to exported processed GPS rows, NOT verified physical receiver fixes.\n    During a false run, keep the last accepted value from BEFORE its onset.\n    Also reject a buffered candidate stamped inside a denied time interval on\n    recovery. No candidate at the first masked sample is accepted. The first\n    grid sample must have a legal fix; this protocol provides explicit warmup.\n    """\n    t = _timeline(t)\n    candidates = np.asarray(candidates)\n    ids = np.asarray(source_indices, dtype=np.int64)\n    st = np.asarray(source_timestamps, dtype=np.float64)\n    available = np.asarray(available, dtype=bool)\n    if (candidates.shape != (len(t), 3) or ids.shape != t.shape\n            or st.shape != t.shape or available.shape != t.shape):\n        raise ValueError("GNSS inputs have incompatible shapes")\n    if not np.isfinite(candidates).all() or not np.isfinite(st).all() or (ids < 0).any():\n        raise ValueError("invalid candidate fixes")\n    if (st > t).any():\n        raise ValueError("R7 resampling selected a future GPS source; refusing silent repair")\n    if blocked_intervals is None:\n        blocked_intervals = [(t[a], t[b] if b < len(t) else np.inf)\n                             for a, b in false_runs(available)]\n    forbidden_source = np.zeros(len(t), dtype=bool)\n    denied_grid = np.zeros(len(t), dtype=bool)\n    for a, b in blocked_intervals:\n        if not a < b:\n            raise ValueError("invalid denied time interval")\n        forbidden_source |= (st >= a) & (st < b)\n        denied_grid |= (t >= a) & (t < b)\n    if not np.array_equal(denied_grid, ~available):\n        raise ValueError("source denial intervals must match the final grid mask")\n    held = np.empty_like(candidates)\n    held_ids = np.empty(len(t), dtype=np.int64)\n    held_st = np.empty(len(t), dtype=np.float64)\n    accepted_at = np.empty(len(t), dtype=np.int64)\n    updates = np.zeros(len(t), dtype=bool)\n    last = None\n    for i in range(len(t)):\n        if available[i] and not forbidden_source[i]:\n            if last is not None and (ids[i] < ids[last] or st[i] < st[last]):\n                raise ValueError("selected source order reversed; refusing silent resampling change")\n            if last is None or ids[i] != ids[last]:\n                last = i\n                updates[i] = True\n            elif not np.array_equal(candidates[i], candidates[last]) or st[i] != st[last]:\n                raise ValueError("same source index has inconsistent timestamp/position")\n        if last is None:\n            raise ValueError("no legal pre-outage fix; provide available GNSS history, never GT")\n        held[i], held_ids[i], held_st[i], accepted_at[i] = candidates[last], ids[last], st[last], last\n    return {"held_gnss": held, "availability_mask": available.copy(),\n            "gnss_update_mask": updates, "held_source_index": held_ids,\n            "held_source_timestamp_s": held_st, "accepted_at_grid_index": accepted_at,\n            "fix_age_s": t - held_st,\n            "rejected_buffered_source_mask": available & forbidden_source}\n\n\ndef constant_velocity_gnss(t, clean, history_s=5.0, min_span_s=1.0):\n    """Frozen causal least-squares XYZ velocity from accepted pre-onset fixes.\n\n    Unique source rows within the trailing history_s (source time), at least\n    two distinct timestamps spanning min_span_s. Otherwise v=0. Anchor at the\n    last legal position AND its source time, extrapolate only in mask=False.\n    Link recovery returns to held-GNSS; no fit to GT or later samples.\n    """\n    t = _timeline(t)\n    if history_s <= 0 or min_span_s <= 0:\n        raise ValueError("positive velocity history/span required")\n    held = clean["held_gnss"]\n    st = clean["held_source_timestamp_s"]\n    update = clean["gnss_update_mask"]\n    pred = held.astype(np.float64).copy()\n    records = []\n    for a, b in false_runs(clean["availability_mask"]):\n        if a == 0:\n            raise ValueError("CV requires a legal pre-onset anchor")\n        hist = np.flatnonzero(update & (np.arange(len(t)) < a)\n                              & (st >= t[a] - history_s) & (st < t[a]))\n        # An exported stream can contain multiple rows with the same timestamp.\n        # Keep the last accepted row per timestamp; do not overweight duplicates.\n        if len(hist):\n            hist = hist[np.r_[np.diff(st[hist]) != 0, True]]\n        velocity = np.zeros(3, dtype=np.float64)\n        span = float(st[hist[-1]] - st[hist[0]]) if len(hist) else 0.0\n        fallback = len(hist) < 2 or span < min_span_s\n        if not fallback:\n            x = st[hist] - st[hist].mean()\n            y = held[hist].astype(np.float64)\n            velocity = (x[:, None] * (y - y.mean(axis=0))).sum(axis=0) / (x @ x)\n        pred[a:b] = held[a - 1] + (t[a:b] - st[a - 1])[:, None] * velocity\n        records.append({"indices_half_open": [a, b], "velocity_m_s": velocity.tolist(),\n                        "history_source_indices": clean["held_source_index"][hist].tolist(),\n                        "history_grid_indices": hist.tolist(), "history_span_s": span,\n                        "history_source_timestamps_s": st[hist].tolist(),\n                        "anchor_source_index": int(clean["held_source_index"][a - 1]),\n                        "anchor_source_timestamp_s": float(st[a - 1]),\n                        "anchor_position_m": held[a - 1].tolist(),\n                        "zero_velocity_fallback": bool(fallback)})\n    return pred, records\n\n\ndef raw_errors(pred, gt):\n    pred, gt = np.asarray(pred, dtype=np.float64), np.asarray(gt, dtype=np.float64)\n    if pred.shape != gt.shape or pred.ndim != 2 or pred.shape[1] != 3:\n        raise ValueError("pred and GT must be matching N x 3 arrays")\n    if not np.isfinite(pred).all() or not np.isfinite(gt).all():\n        raise ValueError("nonfinite prediction/GT")\n    delta = pred - gt\n    return {"error_xyz_m": delta, "error_3d_m": np.linalg.norm(delta, axis=1),\n            "error_horizontal_m": np.linalg.norm(delta[:, :2], axis=1),\n            "error_vertical_signed_m": delta[:, 2]}\n\n\ndef raw_metrics(pred, gt, available):\n    """Sample-weighted RAW errors; no rigid alignment, fit, SD, or CI.\n\n    All unavailable points for an outage scenario (one episode here); the whole\n    common trace for control. Final errors always mean last unavailable sample,\n    and are None for control. Horizontal=existing first two project axes.\n    """\n    errors = raw_errors(pred, gt)\n    available = np.asarray(available, dtype=bool)\n    if available.shape != (len(pred),):\n        raise ValueError("metric mask shape mismatch")\n    missing = np.flatnonzero(~available)\n    chosen = missing if len(missing) else np.arange(len(pred))\n    if not len(chosen):\n        raise ValueError("empty metric support")\n    d = errors["error_xyz_m"][chosen]\n    row = {"metric_scope": "outage_only" if len(missing) else "whole_common_trace",\n           "n_evaluated": len(chosen), "rmse_3d_m": float(np.sqrt(np.mean(np.sum(d * d, axis=1)))),\n           "rmse_horizontal_m": float(np.sqrt(np.mean(np.sum(d[:, :2] ** 2, axis=1)))),\n           "rmse_vertical_m": float(np.sqrt(np.mean(d[:, 2] ** 2))),\n           "final_unavailable_error_3d_m": None, "final_unavailable_error_horizontal_m": None,\n           "final_unavailable_abs_vertical_m": None}\n    if len(missing):\n        last = missing[-1]\n        row.update(final_unavailable_error_3d_m=float(errors["error_3d_m"][last]),\n                   final_unavailable_error_horizontal_m=float(errors["error_horizontal_m"][last]),\n                   final_unavailable_abs_vertical_m=float(abs(errors["error_vertical_signed_m"][last])))\n    return row\n', 'src/diagnostics/run_clean_outages.py': '"""Inference-only runner for a pinned R7 export; invoked manually on Kaggle.\n\nDoes not import train.py/evaluate.py, append R7 results, tune, or train.\nOnly writes a NEW outputs/diagnostic_clean_outages directory.\n"""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport platform\nimport shutil\nimport sys\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom diagnostics.clean_outages import (\n    availability_from_blocks, clean_gnss, common_validation_indices,\n    constant_velocity_gnss, make_scenarios, raw_errors, raw_metrics,\n)\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\\n")\n\n\ndef verify_r7(root, config):\n    """Check exact file bytes BEFORE loading any checkpoint."""\n    root = Path(root)\n    verified = {}\n    for relative, expected in config["r7"]["artifacts"].items():\n        path = root / relative\n        if not path.is_file():\n            raise FileNotFoundError(f"Missing R7 artifact: {path}")\n        if path.stat().st_size != expected["bytes"] or sha256_file(path) != expected["sha256"]:\n            raise ValueError(f"R7 identity mismatch: {path}; expected {expected[\'sha256\']}")\n        verified[relative] = {**expected, "resolved_path": str(path.resolve())}\n    return verified\n\n\ndef discover_r7(input_root, config):\n    roots = sorted({p.parents[3] for p in Path(input_root).rglob("drone/data/processed/t_imu.npy")})\n    matches, rejected = [], []\n    for root in roots:\n        try:\n            verified = verify_r7(root, config)\n            matches.append((root, verified))\n        except (ValueError, FileNotFoundError) as error:\n            rejected.append(str(error))\n    if not matches:\n        detail = "\\n".join(rejected) or "No export with drone/data/processed/t_imu.npy was found."\n        raise FileNotFoundError(\n            "Attach the saved output of shapok/drone-nav-full-run version 7 "\n            "(output ID 352644733), retaining its directory structure.\\n" + detail\n            + "\\nRequired paths:\\n" + "\\n".join(config["r7"]["artifacts"]))\n    if len(matches) != 1:\n        raise ValueError("Multiple matching R7 mounts; pass --r7-root explicitly: "\n                         + ", ".join(str(x[0]) for x in matches))\n    return matches[0]\n\n\ndef prepare_validation(root, config):\n    """Keep R7 frames, channel order, float32 features and GPS searchsorted.\n\n    GPS order is intentionally NOT silently repaired. Record reversals; clean\n    GNSS construction refuses future or reversed selected sources. GT is\n    interpolated only inside original support, on the common validation grid.\n    """\n    data = Path(root) / "drone/data/processed"\n    arrays = {name: np.load(data / (name + ".npy"), allow_pickle=False)\n              for name in ("t_imu", "imu", "t_gps", "gps_pos", "t_gt", "gt_pos")}\n    for name, array in arrays.items():\n        if not np.isfinite(array).all():\n            raise ValueError(f"Nonfinite R7 input: {name}")\n    t, tg, t_gt = arrays["t_imu"], arrays["t_gps"], arrays["t_gt"]\n    if arrays["imu"].shape != (len(t), 6) or arrays["gps_pos"].shape != (len(tg), 3):\n        raise ValueError("Unexpected R7 input shapes")\n    if arrays["gt_pos"].shape != (len(t_gt), 3):\n        raise ValueError("Unexpected GT shape")\n    split = config["split"]\n    if split["name"] != "validation" or len(t) != split["expected_total_imu"]:\n        raise ValueError("This locked diagnostic supports only the R7 validation split")\n    indices, support = common_validation_indices(\n        t, t_gt, split["train_fraction"], split["val_fraction"], config["window"])\n    if (support["original_split_indices_half_open"] != split["expected_original_indices"]\n            or support["common_indices_half_open"] != split["expected_common_indices"]):\n        raise ValueError("Unexpected split/support; do not silently change locked protocol")\n    common_t = t[indices].astype(np.float64)\n    # EXACT existing R7 numeric resampling, including its unsorted-source risk.\n    source = np.clip(np.searchsorted(tg, common_t, side="right") - 1, 0, len(tg) - 1)\n    gps = arrays["gps_pos"][source].astype(np.float32)\n    gt = np.stack([np.interp(common_t, t_gt, arrays["gt_pos"][:, axis])\n                   for axis in range(3)], axis=1).astype(np.float32)\n    support.update(gps_timestamp_reversal_count=int(np.sum(np.diff(tg) < 0)),\n                   selected_future_source_count=int(np.sum(tg[source] > common_t)),\n                   selected_source_time_reversal_count=int(np.sum(np.diff(tg[source]) < 0)),\n                   imu_dt_min_s=float(np.diff(common_t).min()),\n                   imu_dt_median_s=float(np.median(np.diff(common_t))),\n                   imu_dt_max_s=float(np.diff(common_t).max()),\n                   source_identity="processed GPS row, not independently verified physical receiver fix")\n    return {"t": common_t, "imu": arrays["imu"][indices].astype(np.float32), "gps": gps,\n            "gt": gt, "source_indices": source, "source_timestamps": tg[source],\n            "original_indices": indices, "support": support}\n\n\ndef infer_windows(model, imu, gps, available, window, batch_windows, device, zero_imu=False):\n    """Reset state per nonoverlapping window, no ground-truth input or reset."""\n    import torch\n    n = len(imu)\n    if n % window or batch_windows < 1:\n        raise ValueError("all methods require the same complete-window prefix")\n    if imu.shape != (n, 6) or gps.shape != (n, 3) or available.shape != (n,):\n        raise ValueError("invalid inference input shapes")\n    imu_windows = np.asarray(imu, dtype=np.float32).reshape(-1, window, 6)\n    gps_windows = np.asarray(gps, dtype=np.float32).reshape(-1, window, 3)\n    mask_windows = np.asarray(available, dtype=np.float32).reshape(-1, window, 1)\n    model.eval()\n    predictions = []\n    with torch.inference_mode():\n        for start in range(0, len(imu_windows), batch_windows):\n            stop = start + batch_windows\n            i = torch.from_numpy(imu_windows[start:stop]).to(device)\n            if zero_imu:\n                i = torch.zeros_like(i)\n            g = torch.from_numpy(gps_windows[start:stop]).to(device)\n            m = torch.from_numpy(mask_windows[start:stop]).to(device)\n            out = model(i, g, m)\n            predictions.append(out.cpu().numpy().reshape(-1, 3))\n    result = np.concatenate(predictions)\n    if result.shape != (n, 3) or not np.isfinite(result).all():\n        raise ValueError("invalid inference result")\n    return result\n\n\ndef unique_rows(rows):\n    keys = [(r["scenario_id"], r["method"], r["seed"]) for r in rows]\n    if len(set(keys)) != len(keys):\n        raise ValueError("duplicate evaluation rows")\n    return rows\n\n\ndef write_csv(path, rows):\n    if not rows:\n        raise ValueError("no rows")\n    with Path(path).open("x", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef reserve_output(path):\n    """No append, resume, force flag, or overwrite of existing diagnostics."""\n    path = Path(path)\n    path.mkdir(parents=True, exist_ok=False)\n    return path\n\n\ndef save_prediction(directory, pred, data, clean, scenario, metadata):\n    """One self-contained bundle per method/scenario; no pickle needed to read."""\n    directory.mkdir()\n    correction = pred.astype(np.float64) - clean["held_gnss"].astype(np.float64)\n    errors = raw_errors(pred, data["gt"])\n    np.savez_compressed(\n        directory / "prediction.npz", pred=pred, gt=data["gt"], timestamps_s=data["t"],\n        original_imu_indices=data["original_indices"],\n        scenario_id=np.array(scenario["scenario_id"]), metadata_json=np.array(json.dumps(metadata)),\n        correction_xyz_m=correction, correction_norm_m=np.linalg.norm(correction, axis=1),\n        **clean, **errors)\n    write_json(directory / "metadata.json", metadata)\n    selected = ~clean["availability_mask"]\n    if not selected.any():\n        selected = np.ones(len(pred), dtype=bool)\n    norms = np.linalg.norm(correction[selected], axis=1)\n    return {"correction_rms_norm_m": float(np.sqrt(np.mean(norms ** 2))),\n            "correction_mean_norm_m": float(norms.mean()),\n            "correction_max_norm_m": float(norms.max())}\n\n\ndef run(config_path, input_root="/kaggle/input", r7_root=None,\n        out_dir="outputs/diagnostic_clean_outages"):\n    import torch\n    from models.fusion_lstm import FusionLSTM\n    from models.fusion_transformer import FusionTransformer\n\n    config_path = Path(config_path)\n    config = json.loads(config_path.read_text())\n    project = Path(__file__).resolve().parents[2]\n    for relative, expected in config["r7"]["model_source_sha256"].items():\n        if sha256_file(project / relative) != expected:\n            raise ValueError(f"R7 architecture source changed: {relative}")\n    if r7_root is None:\n        root, verified = discover_r7(input_root, config)\n    else:\n        root = Path(r7_root)\n        verified = verify_r7(root, config)\n    out_path = Path(out_dir).resolve()\n    if root.resolve() == out_path or root.resolve() in out_path.parents:\n        raise ValueError("diagnostic output must be outside the R7 export")\n    data = prepare_validation(root, config)\n    scenarios = make_scenarios(data["t"], config["protocol"])\n    locked_path = config_path.with_name("diagnostic_clean_outages_manifest.json")\n    locked = json.loads(locked_path.read_text())\n    t_hash = hashlib.sha256(data["t"].astype("<f8").tobytes()).hexdigest()\n    if scenarios != locked["scenarios"] or t_hash != locked["common_timestamps_float64_le_sha256"]:\n        raise ValueError("scenario manifest differs from the timestamp-only preflight")\n    if locked["config_sha256"] != sha256_file(config_path):\n        raise ValueError("configuration differs from the locked preflight")\n    if len({m["id"] for m in config["methods"]}) != len(config["methods"]):\n        raise ValueError("duplicate configured methods")\n\n    torch.manual_seed(config["inference"]["seed"])\n    np.random.seed(config["inference"]["seed"])\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(config["inference"]["seed"])\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    models = {}\n    for method in config["methods"]:\n        checkpoint = method.get("checkpoint")\n        if checkpoint and checkpoint not in models:\n            model = (FusionTransformer(ablation=method["ablation"])\n                     if method["architecture"] == "fusion_transformer" else FusionLSTM())\n            state = torch.load(root / checkpoint, map_location="cpu", weights_only=True)\n            model.load_state_dict(state, strict=True)\n            models[checkpoint] = model.to(device).eval()\n    out = reserve_output(out_path)\n    write_json(out / "status.json", {"status": "running", "started_utc": datetime.now(timezone.utc).isoformat()})\n    try:\n        shutil.copyfile(config_path, out / "config.json")\n        shutil.copyfile(locked_path, out / "locked_protocol_manifest.json")\n        snapshot = out / "code"\n        for relative in ("src/diagnostics/__init__.py", "src/diagnostics/clean_outages.py",\n                         "src/diagnostics/run_clean_outages.py", *config["r7"]["model_source_sha256"]):\n            destination = snapshot / relative\n            destination.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copyfile(project / relative, destination)\n        provenance = {\n            "protocol_id": config["protocol_id"], "r7": config["r7"], "verified_artifacts": verified,\n            "config_sha256": sha256_file(config_path), "locked_manifest_sha256": sha256_file(locked_path),\n            "executed_source_sha256": {str(p.relative_to(snapshot)): sha256_file(p)\n                                       for p in sorted(snapshot.rglob("*.py"))},\n            "environment": {"python": platform.python_version(), "numpy": np.__version__,\n                            "torch": torch.__version__, "cuda": torch.version.cuda, "device": device,\n                            "device_name": torch.cuda.get_device_name(0) if device == "cuda" else platform.processor()},\n            "validation_support": data["support"],\n            "claim_limits": ["validation diagnostic, one seed and one flight",\n                             "Transformer is noncausal within each 192-sample window",\n                             "original coordinate agreement and IMU synchronization unresolved",\n                             "source provenance identifies exported GPS rows, not physical receiver packets",\n                             "no bitwise guarantee across Torch/CUDA versions"],\n        }\n        write_json(out / "provenance.json", provenance)\n        write_json(out / "protocol_manifest.json", {**locked, "runtime_support": data["support"]})\n        rows, pairs = [], []\n        for scenario in scenarios:\n            sid = scenario["scenario_id"]\n            scenario_dir = out / sid\n            scenario_dir.mkdir()\n            available = availability_from_blocks(len(data["t"]), scenario["unavailable_index_runs"])\n            clean = clean_gnss(data["t"], data["gps"], data["source_indices"],\n                               data["source_timestamps"], available, scenario["source_blocked_intervals_s"])\n            if sid == "val_control" and not np.array_equal(clean["held_gnss"], data["gps"]):\n                raise ValueError("control altered original R7 GPS values")\n            cv, cv_manifest = constant_velocity_gnss(data["t"], clean, **{\n                k: config["constant_velocity"][k] for k in ("history_s", "min_span_s")})\n            write_json(scenario_dir / "scenario.json", {**scenario, "constant_velocity_fit": cv_manifest})\n            np.savez_compressed(scenario_dir / "inputs.npz", timestamps_s=data["t"], imu=data["imu"],\n                                gt=data["gt"], original_imu_indices=data["original_indices"], **clean)\n            scenario_rows = {}\n            for method in config["methods"]:\n                mid = method["id"]\n                if mid == "held_gnss":\n                    pred = clean["held_gnss"].copy()\n                elif mid == "constant_velocity_gnss":\n                    pred = cv\n                else:\n                    pred = infer_windows(models[method["checkpoint"]], data["imu"], clean["held_gnss"],\n                                         available, config["window"], config["inference"]["batch_windows"],\n                                         device, zero_imu=method.get("zero_imu", False))\n                checkpoint = method.get("checkpoint")\n                metadata = {"scenario": scenario, "method": method,\n                            "checkpoint_sha256": verified[checkpoint]["sha256"] if checkpoint else None,\n                            "protocol_id": config["protocol_id"], "r7_output_id": config["r7"]["output_id"],\n                            "config_sha256": provenance["config_sha256"], "metric_definition": config["metrics"],\n                            "model_semantics": config["inference"], "claim_limits": provenance["claim_limits"]}\n                correction = save_prediction(scenario_dir / mid, pred, data, clean, scenario, metadata)\n                row = {"protocol_id": config["protocol_id"], "scenario_id": sid, "split": "validation",\n                       "episode_id": scenario["episode_id"], "requested_duration_s": scenario["requested_duration_s"],\n                       "method": mid, "kind": method["kind"], "seed": method.get("seed"),\n                       "checkpoint_sha256": metadata["checkpoint_sha256"],\n                       "mask_uint8_sha256": scenario["mask_uint8_sha256"], "n_common": len(data["t"]),\n                       **raw_metrics(pred, data["gt"], available), **correction,\n                       "prediction_file": f"{sid}/{mid}/prediction.npz"}\n                rows.append(row)\n                scenario_rows[mid] = row\n                print(f"{sid} / {mid}: raw {row[\'metric_scope\']} RMSE3D={row[\'rmse_3d_m\']:.6f} m")\n            full = scenario_rows["fusion_full"]\n            pair = {"scenario_id": sid, "episode_id": scenario["episode_id"],\n                    "requested_duration_s": scenario["requested_duration_s"], "split": "validation",\n                    "metric_scope": full["metric_scope"], "n_evaluated": full["n_evaluated"],\n                    "seed": 0, "delta_definition": "full minus comparator; negative means smaller full RMSE"}\n            for other in ("fusion_gps_only", "held_gnss", "constant_velocity_gnss", "fusion_lstm"):\n                pair[f"full_minus_{other}_rmse_3d_m"] = full["rmse_3d_m"] - scenario_rows[other]["rmse_3d_m"]\n            pair["sensitivity_zero_imu_minus_full_rmse_3d_m"] = (\n                scenario_rows["fusion_full_zero_imu"]["rmse_3d_m"] - full["rmse_3d_m"])\n            pairs.append(pair)\n        unique_rows(rows)\n        if len(rows) != len(scenarios) * len(config["methods"]):\n            raise ValueError("incomplete evaluation matrix")\n        write_csv(out / "summary.csv", rows)\n        write_csv(out / "paired_differences.csv", pairs)\n        write_json(out / "status.json", {"status": "complete", "completed_utc": datetime.now(timezone.utc).isoformat(),\n                                         "evaluation_rows": len(rows), "scenarios": len(scenarios),\n                                         "training_performed": False, "test_evaluated": False})\n        catalog = {str(p.relative_to(out)): {"bytes": p.stat().st_size, "sha256": sha256_file(p)}\n                   for p in sorted(out.rglob("*")) if p.is_file()}\n        write_json(out / "artifact_manifest.json", catalog)\n        return out\n    except Exception as error:\n        write_json(out / "status.json", {"status": "failed", "error": str(error),\n                                         "training_performed": False, "test_evaluated": False})\n        raise\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--config", required=True)\n    parser.add_argument("--input-root", default="/kaggle/input")\n    parser.add_argument("--r7-root")\n    parser.add_argument("--out-dir", default="outputs/diagnostic_clean_outages")\n    args = parser.parse_args()\n    run(args.config, args.input_root, args.r7_root, args.out_dir)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/models/fusion_transformer.py': '"""Dual-branch cross-attention fusion Transformer (proposed method).\n\nOne branch encodes the IMU stream (accel+gyro, always present, full rate).\nOne branch encodes the GPS stream (position + an explicit availability\nmask channel, so "GPS missing" is a signal the model sees, not silently\ninterpolated away). Cross-attention lets the IMU branch pull information\nfrom GPS when it\'s available and fall back to pure inertial reasoning\nwhen it isn\'t. A shared head regresses 3D ENU position at every timestep.\n"""\nimport math\n\nimport torch\nimport torch.nn as nn\n\n\nclass SinusoidalPositionalEncoding(nn.Module):\n    def __init__(self, d_model, max_len=4096):\n        super().__init__()\n        pe = torch.zeros(max_len, d_model)\n        pos = torch.arange(max_len).unsqueeze(1).float()\n        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))\n        pe[:, 0::2] = torch.sin(pos * div)\n        pe[:, 1::2] = torch.cos(pos * div)\n        self.register_buffer("pe", pe.unsqueeze(0))\n\n    def forward(self, x):\n        return x + self.pe[:, : x.size(1)]\n\n\nclass CrossAttentionBlock(nn.Module):\n    """query stream attends over key/value stream, residual + FFN."""\n\n    def __init__(self, d_model, n_heads, ff_mult=4, dropout=0.1):\n        super().__init__()\n        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)\n        self.norm1 = nn.LayerNorm(d_model)\n        self.ff = nn.Sequential(\n            nn.Linear(d_model, d_model * ff_mult),\n            nn.GELU(),\n            nn.Linear(d_model * ff_mult, d_model),\n        )\n        self.norm2 = nn.LayerNorm(d_model)\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, query, kv, return_attn=False):\n        attn_out, attn_weights = self.attn(query, kv, kv, need_weights=return_attn,\n                                            average_attn_weights=True)\n        x = self.norm1(query + self.dropout(attn_out))\n        x = self.norm2(x + self.dropout(self.ff(x)))\n        return (x, attn_weights) if return_attn else (x, None)\n\n\nclass FusionTransformer(nn.Module):\n    """\n    Inputs:\n      imu:      (B, T, 6)  -- accel_xyz, gyro_xyz, full-rate IMU grid\n      gps:      (B, T, 3)  -- ENU position resampled/held onto the IMU grid\n      gps_mask: (B, T, 1)  -- 1.0 where GPS is actually available at t, else 0.0\n    Output:\n      pos_pred: (B, T, 3)  -- predicted ENU position at every IMU timestep\n    """\n\n    ABLATIONS = ("full", "no_cross_attn", "imu_only", "gps_only")\n\n    def __init__(self, d_model=128, n_heads=4, n_layers=2, dropout=0.1, ablation="full"):\n        super().__init__()\n        assert ablation in self.ABLATIONS, ablation\n        self.ablation = ablation\n\n        self.imu_embed = nn.Linear(6, d_model)\n        self.gps_embed = nn.Linear(4, d_model)  # position (3) + mask (1)\n        self.pos_enc = SinusoidalPositionalEncoding(d_model)\n\n        self.imu_self_layers = nn.ModuleList([\n            nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=d_model * 4,\n                                        dropout=dropout, batch_first=True)\n            for _ in range(n_layers)\n        ])\n        if ablation == "no_cross_attn":\n            # same parameter budget in the fusion step, no attention: a\n            # concat+linear merge instead of cross-attention. Isolates what\n            # cross-attention itself buys over a static fusion.\n            self.concat_fuse = nn.ModuleList([\n                nn.Sequential(nn.Linear(d_model * 2, d_model), nn.GELU(), nn.LayerNorm(d_model))\n                for _ in range(n_layers)\n            ])\n        else:\n            self.cross_layers = nn.ModuleList([\n                CrossAttentionBlock(d_model, n_heads, dropout=dropout) for _ in range(n_layers)\n            ])\n\n        self.head = nn.Sequential(\n            nn.Linear(d_model, d_model // 2),\n            nn.GELU(),\n            nn.Linear(d_model // 2, 3),\n        )\n\n    def forward(self, imu, gps, gps_mask, return_attn=False):\n        if self.ablation == "gps_only":\n            imu = torch.zeros_like(imu)\n        gps_in = torch.cat([gps, gps_mask], dim=-1)\n        if self.ablation == "imu_only":\n            gps_in = torch.zeros_like(gps_in)\n\n        h_imu = self.pos_enc(self.imu_embed(imu))\n        h_gps = self.pos_enc(self.gps_embed(gps_in))\n\n        for layer in self.imu_self_layers:\n            h_imu = layer(h_imu)\n\n        attn_maps = []\n        if self.ablation == "no_cross_attn":\n            for fuse in self.concat_fuse:\n                h_imu = fuse(torch.cat([h_imu, h_gps], dim=-1))\n        else:\n            for cross in self.cross_layers:\n                h_imu, attn = cross(h_imu, h_gps, return_attn=return_attn)\n                if return_attn:\n                    attn_maps.append(attn)\n\n        delta = self.head(h_imu)\n        if self.ablation == "imu_only":\n            # No GPS numeric value anywhere in the output, not even as a\n            # residual base -- otherwise this ablation would still see\n            # GPS through the back door and the comparison would be\n            # meaningless (this is exactly what happened before the fix:\n            # imu_only scored identically to full because it was still\n            # just correcting raw GPS).\n            pos_pred = delta\n        else:\n            # residual on top of the last-known (held) GPS position keeps\n            # the regression target small and centered, easier to fit\n            # than raw ENU.\n            pos_pred = gps + delta\n        return (pos_pred, attn_maps) if return_attn else pos_pred\n\n\nif __name__ == "__main__":\n    torch.manual_seed(0)\n    B, T = 4, 96\n    model = FusionTransformer(d_model=64, n_heads=4, n_layers=2)\n    model.eval()  # dropout off so the two forward passes below are comparable\n\n    imu = torch.randn(B, T, 6)\n    gps = torch.randn(B, T, 3)\n    gps_mask = (torch.rand(B, T, 1) > 0.7).float()\n\n    out = model(imu, gps, gps_mask)\n    assert out.shape == (B, T, 3), out.shape\n    assert torch.isfinite(out).all()\n\n    out2, attn_maps = model(imu, gps, gps_mask, return_attn=True)\n    # need_weights=True forces a different (non-fused) attention kernel in\n    # PyTorch, so this only matches to float32 kernel-switch precision.\n    assert torch.allclose(out, out2, atol=1e-5)\n    assert len(attn_maps) == 2 and attn_maps[0].shape[0] == B\n\n    n_params = sum(p.numel() for p in model.parameters())\n\n    for ablation in ("no_cross_attn", "imu_only", "gps_only"):\n        m = FusionTransformer(d_model=64, n_heads=4, n_layers=2, ablation=ablation).eval()\n        o = m(imu, gps, gps_mask)\n        assert o.shape == (B, T, 3) and torch.isfinite(o).all(), ablation\n\n    print(f"fusion_transformer ok | params={n_params/1e3:.1f}K | out={tuple(out.shape)} | "\n          f"ablations={FusionTransformer.ABLATIONS}")\n', 'src/models/fusion_lstm.py': '"""Dual-branch fusion, LSTM backbone -- ablation arm answering "why a\nTransformer and not a recurrent model" for the paper. Same I/O contract\nas FusionTransformer so train.py/evaluate.py treat both uniformly; fusion\nhere is a learned gate instead of cross-attention (no attention weights\nto extract, so this model is not used for the Figure 3 XAI plot).\n"""\nimport torch\nimport torch.nn as nn\n\n\nclass FusionLSTM(nn.Module):\n    """\n    Inputs:\n      imu:      (B, T, 6)\n      gps:      (B, T, 3)\n      gps_mask: (B, T, 1)\n    Output:\n      pos_pred: (B, T, 3)\n    """\n\n    def __init__(self, hidden=128, n_layers=2, dropout=0.1):\n        super().__init__()\n        self.imu_lstm = nn.LSTM(6, hidden, n_layers, batch_first=True,\n                                 dropout=dropout if n_layers > 1 else 0.0)\n        self.gps_lstm = nn.LSTM(4, hidden, n_layers, batch_first=True,\n                                 dropout=dropout if n_layers > 1 else 0.0)\n        self.gate = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.Sigmoid())\n        self.head = nn.Sequential(\n            nn.Linear(hidden, hidden // 2),\n            nn.GELU(),\n            nn.Linear(hidden // 2, 3),\n        )\n\n    def forward(self, imu, gps, gps_mask):\n        gps_in = torch.cat([gps, gps_mask], dim=-1)\n        h_imu, _ = self.imu_lstm(imu)\n        h_gps, _ = self.gps_lstm(gps_in)\n\n        g = self.gate(torch.cat([h_imu, h_gps], dim=-1))\n        fused = g * h_imu + (1 - g) * h_gps\n\n        delta = self.head(fused)\n        return gps + delta\n\n\nif __name__ == "__main__":\n    torch.manual_seed(0)\n    B, T = 4, 96\n    model = FusionLSTM(hidden=64, n_layers=2)\n    model.eval()\n\n    imu = torch.randn(B, T, 6)\n    gps = torch.randn(B, T, 3)\n    gps_mask = (torch.rand(B, T, 1) > 0.7).float()\n\n    out = model(imu, gps, gps_mask)\n    assert out.shape == (B, T, 3), out.shape\n    assert torch.isfinite(out).all()\n\n    n_params = sum(p.numel() for p in model.parameters())\n    print(f"fusion_lstm ok | params={n_params/1e3:.1f}K | out={tuple(out.shape)}")\n', 'tests/test_clean_outages.py': '"""Small synthetic tests only: no R7 inference, training or Kaggle jobs.\n\nRun: python3 -B -m unittest discover -s tests -p \'test_clean_outages.py\' -v\n"""\nimport hashlib\nimport contextlib\nimport csv\nimport io\nimport json\nimport sys\nimport tempfile\nimport unittest\nfrom unittest.mock import patch\nfrom pathlib import Path\n\nimport numpy as np\n\nPROJECT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(PROJECT / "src"))\n\nfrom diagnostics.clean_outages import (\n    availability_from_blocks, clean_gnss, common_validation_indices,\n    constant_velocity_gnss, false_runs, make_scenarios, mask_sha256,\n    raw_errors, raw_metrics,\n)\nfrom diagnostics.run_clean_outages import (\n    discover_r7, infer_windows, prepare_validation, reserve_output,\n    run, save_prediction, unique_rows, verify_r7, write_csv,\n)\n\n\nclass CleanGeneratorTests(unittest.TestCase):\n    def setUp(self):\n        self.t = np.arange(20, dtype=float)\n        self.gps = np.column_stack((self.t, self.t * 2, -self.t)).astype(np.float32)\n        self.ids = np.arange(20)\n\n    def clean(self, blocks, gps=None):\n        return clean_gnss(self.t, self.gps if gps is None else gps, self.ids,\n                          self.t - 0.1, availability_from_blocks(20, blocks))\n\n    def assert_constant_runs(self, clean):\n        for a, b in false_runs(clean["availability_mask"]):\n            self.assertFalse(clean["gnss_update_mask"][a:b].any())\n            np.testing.assert_array_equal(clean["held_gnss"][a:b],\n                                          np.repeat(self.gps[a - 1:a], b - a, axis=0))\n            self.assertTrue((clean["held_source_index"][a:b] == a - 1).all())\n            self.assertTrue((clean["held_source_timestamp_s"][a:b] < self.t[a]).all())\n            self.assertTrue((clean["accepted_at_grid_index"][a:b] < a).all())\n\n    def test_single_outage_strict_previous_fix(self):\n        result = self.clean([(5, 9)])\n        self.assert_constant_runs(result)\n        np.testing.assert_array_equal(result["held_gnss"][5], self.gps[4])\n        self.assertFalse(np.array_equal(result["held_gnss"][5], self.gps[5]))\n\n    def test_overlaps_both_orders_same_final_mask_and_values(self):\n        left = self.clean([(5, 10), (8, 14)])\n        right = self.clean([(8, 14), (5, 10)])\n        for key in left:\n            np.testing.assert_array_equal(left[key], right[key])\n        self.assertEqual(false_runs(left["availability_mask"]), [(5, 14)])\n        self.assert_constant_runs(left)\n\n    def test_nested_blocks(self):\n        left = self.clean([(4, 16), (7, 10)])\n        right = self.clean([(7, 10), (4, 16)])\n        self.assert_constant_runs(left)\n        np.testing.assert_array_equal(left["held_gnss"], right["held_gnss"])\n\n    def test_outage_to_record_end(self):\n        self.assert_constant_runs(self.clean([(6, 20)]))\n\n    def test_adjacent_blocks_are_one_outage(self):\n        result = self.clean([(4, 8), (8, 12)])\n        self.assertEqual(false_runs(result["availability_mask"]), [(4, 12)])\n        self.assert_constant_runs(result)\n\n    def test_masked_values_cannot_affect_inputs(self):\n        blocks = [(5, 9), (12, 20)]\n        changed = self.gps.copy()\n        changed[~availability_from_blocks(20, blocks)] = 999999\n        left, right = self.clean(blocks), self.clean(blocks, changed)\n        np.testing.assert_array_equal(left["held_gnss"], right["held_gnss"])\n\n    def test_control_preserves_input(self):\n        np.testing.assert_array_equal(self.clean([])["held_gnss"], self.gps)\n\n    def test_cannot_fill_initial_missing_with_gt_or_first_masked_fix(self):\n        with self.assertRaisesRegex(ValueError, "no legal pre-outage fix"):\n            self.clean([(0, 4)])\n\n    def test_repeated_source_is_not_a_new_measurement(self):\n        ids = self.ids // 2\n        result = clean_gnss(self.t, self.gps[ids], ids, ids * 2 - 0.1, np.ones(20, bool))\n        np.testing.assert_array_equal(np.flatnonzero(result["gnss_update_mask"]), np.arange(0, 20, 2))\n        self.assertTrue(result["availability_mask"].all())\n\n    def test_buffered_outage_fix_rejected_at_recovery(self):\n        t = np.array([0., 1., 2., 3., 4.])\n        st = np.array([0., 0.9, 1.9, 2.4, 3.9])\n        gps = np.repeat(np.arange(5.)[:, None], 3, axis=1)\n        result = clean_gnss(t, gps, np.arange(5), st, np.array([1, 1, 0, 1, 1], bool), [(2., 2.5)])\n        self.assertEqual(result["held_source_index"].tolist(), [0, 1, 1, 1, 4])\n        self.assertTrue(result["rejected_buffered_source_mask"][3])\n        self.assertTrue(result["availability_mask"][3])\n        self.assertFalse(result["gnss_update_mask"][3])\n\n    def test_future_selected_source_fails(self):\n        with self.assertRaisesRegex(ValueError, "future GPS"):\n            clean_gnss(self.t, self.gps, self.ids, self.t + 0.01, np.ones(20, bool))\n\n    def test_reversed_selected_source_fails(self):\n        ids = self.ids.copy()\n        ids[9] = 7\n        with self.assertRaisesRegex(ValueError, "order reversed"):\n            clean_gnss(self.t, self.gps[ids], ids, self.t[ids] - 0.1, np.ones(20, bool))\n\n    def test_source_intervals_must_match_final_mask(self):\n        with self.assertRaisesRegex(ValueError, "match the final grid mask"):\n            clean_gnss(self.t, self.gps, self.ids, self.t - 0.1,\n                        availability_from_blocks(20, [(3, 8)]), [(3, 7)])\n\n\nclass ProtocolTests(unittest.TestCase):\n    def test_gt_support_and_tail_crop_shared(self):\n        indices, info = common_validation_indices(np.arange(100.), np.array([52., 66.]), .5, .2, 4)\n        np.testing.assert_array_equal(indices, np.arange(52, 64))\n        self.assertEqual(info["excluded_outside_gt_support_n"], 5)\n        self.assertEqual(info["discarded_incomplete_window_n"], 3)\n        self.assertEqual(info["common_indices_half_open"], [52, 64])\n\n    def test_irregular_time_duration_and_common_deterministic_starts(self):\n        t = 1000 + np.cumsum(np.tile([.11, .21, .33, .17], 1200))\n        config = {"durations_s": [10, 30, 60], "start_fractions": [.25, .5, .75],\n                  "warmup_s": 30, "recovery_s": 20, "generator_seed": 0}\n        scenarios = make_scenarios(t, config)\n        self.assertEqual(scenarios, make_scenarios(t.copy(), config))\n        self.assertEqual(len(scenarios), 10)\n        for scenario in scenarios[1:]:\n            a, b = scenario["unavailable_index_runs"][0]\n            d = scenario["requested_duration_s"]\n            self.assertLess(t[b - 1] - t[a], d)\n            self.assertGreaterEqual(t[b] - t[a], d)\n            self.assertGreaterEqual(t[a] - t[0], 30)\n            self.assertGreaterEqual(t[-1] - t[b], 20)\n            self.assertNotEqual(b - a, round(d / .1))\n        for episode in (1, 2, 3):\n            self.assertEqual(len({s["onset_s"] for s in scenarios[1:] if s["episode_id"] == episode}), 1)\n\n    def test_too_short_protocol_fails_instead_of_changing_starts(self):\n        config = {"durations_s": [60], "start_fractions": [.25, .5, .75],\n                  "warmup_s": 30, "recovery_s": 20, "generator_seed": 0}\n        with self.assertRaisesRegex(ValueError, "too short"):\n            make_scenarios(np.arange(100.), config)\n\n    def test_locked_manifest_identity_masks_and_common_counts(self):\n        config_path = PROJECT / "configs/diagnostic_clean_outages.json"\n        manifest = json.loads((PROJECT / "configs/diagnostic_clean_outages_manifest.json").read_text())\n        self.assertEqual(manifest["config_sha256"], hashlib.sha256(config_path.read_bytes()).hexdigest())\n        self.assertEqual(len(manifest["scenarios"]), 10)\n        for scenario in manifest["scenarios"]:\n            mask = availability_from_blocks(scenario["n_common"], scenario["unavailable_index_runs"])\n            self.assertEqual(scenario["n_common"], 2688)\n            self.assertEqual(mask_sha256(mask), scenario["mask_uint8_sha256"])\n            self.assertEqual(int((~mask).sum()), scenario["n_unavailable"])\n\n    def test_prepare_validation_does_not_extrapolate_gt(self):\n        with tempfile.TemporaryDirectory() as temporary:\n            data = Path(temporary) / "drone/data/processed"\n            data.mkdir(parents=True)\n            arrays = {"t_imu": np.arange(100.), "imu": np.zeros((100, 6)),\n                      "t_gps": np.arange(100.) - .1, "gps_pos": np.zeros((100, 3)),\n                      "t_gt": np.array([52., 66.]), "gt_pos": np.array([[0, 0, 0], [14, 28, 42]])}\n            for name, array in arrays.items():\n                np.save(data / (name + ".npy"), array)\n            config = {"window": 4, "split": {"name": "validation", "train_fraction": .5,\n                      "val_fraction": .2, "expected_total_imu": 100,\n                      "expected_original_indices": [50, 70], "expected_common_indices": [52, 64]}}\n            result = prepare_validation(temporary, config)\n            np.testing.assert_array_equal(result["t"], np.arange(52., 64.))\n            np.testing.assert_allclose(result["gt"][-1], [11, 22, 33])\n\n\nclass VelocityAndMetricsTests(unittest.TestCase):\n    def test_cv_velocity_anchor_source_time_and_future_independence(self):\n        t = np.arange(25.)\n        st = t - .25\n        gps = np.column_stack((2 * st, -st, .5 * st)).astype(np.float32)\n        mask = availability_from_blocks(len(t), [(10, 18)])\n        clean = clean_gnss(t, gps, np.arange(len(t)), st, mask)\n        pred, records = constant_velocity_gnss(t, clean)\n        np.testing.assert_allclose(records[0]["velocity_m_s"], [2, -1, .5])\n        np.testing.assert_allclose(pred[10:18], np.column_stack((2 * t, -t, .5 * t))[10:18])\n        changed = gps.copy()\n        changed[10:] = 1e6\n        pred_changed, records_changed = constant_velocity_gnss(t, clean_gnss(t, changed, np.arange(len(t)), st, mask))\n        np.testing.assert_array_equal(pred[10:18], pred_changed[10:18])\n        self.assertEqual(records, records_changed)\n        self.assertTrue(all(x < 10 for x in records[0]["history_grid_indices"]))\n\n    def test_cv_fallback_with_insufficient_unique_history(self):\n        t = np.arange(20.)\n        clean = clean_gnss(t, np.ones((20, 3)), np.zeros(20, int), np.zeros(20),\n                           availability_from_blocks(20, [(4, 10)]))\n        pred, records = constant_velocity_gnss(t, clean)\n        self.assertTrue(records[0]["zero_velocity_fallback"])\n        np.testing.assert_array_equal(pred, np.ones((20, 3)))\n\n    def test_cv_deduplicates_source_timestamps(self):\n        t = np.arange(12.)\n        st = np.floor(t / 2) * 2\n        pos = np.repeat(st[:, None], 3, axis=1)\n        clean = clean_gnss(t, pos, np.arange(12), st, availability_from_blocks(12, [(8, 12)]))\n        _, records = constant_velocity_gnss(t, clean)\n        self.assertEqual(records[0]["history_source_timestamps_s"], [4., 6.])\n        np.testing.assert_allclose(records[0]["velocity_m_s"], [1, 1, 1])\n\n    def test_raw_metrics_outage_only_and_last_unavailable(self):\n        pred = np.array([[999, 999, 999], [3, 4, 12], [0, 0, 5], [999, 999, 999.]])\n        row = raw_metrics(pred, np.zeros_like(pred), np.array([1, 0, 0, 1], bool))\n        self.assertEqual(row["n_evaluated"], 2)\n        self.assertAlmostEqual(row["rmse_3d_m"], np.sqrt(97))\n        self.assertAlmostEqual(row["rmse_horizontal_m"], np.sqrt(12.5))\n        self.assertAlmostEqual(row["rmse_vertical_m"], np.sqrt(84.5))\n        self.assertEqual(row["final_unavailable_error_3d_m"], 5)\n        self.assertEqual(row["final_unavailable_error_horizontal_m"], 0)\n        self.assertEqual(row["final_unavailable_abs_vertical_m"], 5)\n\n    def test_control_raw_no_alignment_and_no_fake_outage_endpoint(self):\n        row = raw_metrics(np.ones((3, 3)), np.zeros((3, 3)), np.ones(3, bool))\n        self.assertAlmostEqual(row["rmse_3d_m"], np.sqrt(3))\n        self.assertEqual(row["metric_scope"], "whole_common_trace")\n        self.assertIsNone(row["final_unavailable_error_3d_m"])\n\n    def test_nonfinite_metrics_rejected(self):\n        with self.assertRaisesRegex(ValueError, "nonfinite"):\n            raw_metrics(np.array([[np.nan, 0, 0]]), np.zeros((1, 3)), np.ones(1, bool))\n\n\nclass ArtifactAndSmokeTests(unittest.TestCase):\n    def test_end_to_end_runner_on_tiny_synthetic_fixture_only(self):\n        """Exercise manifests, 60 unique rows and bundles, never a real checkpoint."""\n        import torch\n        from models.fusion_transformer import FusionTransformer\n        from models.fusion_lstm import FusionLSTM\n        torch.manual_seed(0)\n        torch.set_num_threads(1)\n        full = FusionTransformer(d_model=8, n_heads=2, n_layers=1, dropout=0)\n        gps_only = FusionTransformer(d_model=8, n_heads=2, n_layers=1, dropout=0, ablation="gps_only")\n        lstm = FusionLSTM(hidden=8, n_layers=1, dropout=0)\n        def synthetic_state(path, **kwargs):\n            name = str(path)\n            return (lstm if "fusion_lstm" in name else gps_only if "gps_only" in name else full).state_dict()\n        with tempfile.TemporaryDirectory() as temporary:\n            base = Path(temporary)\n            root = base / "r7_fixture"\n            data_dir = root / "drone/data/processed"\n            data_dir.mkdir(parents=True)\n            t = np.arange(100.)\n            arrays = {"t_imu": t, "imu": np.zeros((100, 6), np.float32),\n                      "t_gps": t - .1, "gps_pos": np.repeat((t - .1)[:, None], 3, axis=1).astype(np.float32),\n                      "t_gt": np.array([0., 99.]), "gt_pos": np.array([[0., 0., 0.], [99., 99., 99.]])}\n            for name, array in arrays.items():\n                np.save(data_dir / (name + ".npy"), array)\n            (data_dir / "meta.json").write_text(\'{}\')\n            config = json.loads((PROJECT / "configs/diagnostic_clean_outages.json").read_text())\n            config.update(protocol_id="synthetic_unit_test_only", window=8)\n            config["split"].update(train_fraction=.5, val_fraction=.3, expected_total_imu=100,\n                                   expected_original_indices=[50, 80], expected_common_indices=[50, 74])\n            config["protocol"].update(durations_s=[1, 2, 3], warmup_s=1, recovery_s=1)\n            for relative in config["r7"]["artifacts"]:\n                path = root / relative\n                if relative.endswith(\'.pt\'):\n                    path.parent.mkdir(parents=True, exist_ok=True)\n                    path.write_bytes(b"synthetic placeholder; loading is mocked; not weights")\n                raw = path.read_bytes()\n                config["r7"]["artifacts"][relative] = {"bytes": len(raw), "sha256": hashlib.sha256(raw).hexdigest()}\n            config_path = base / "config.json"\n            config_path.write_text(json.dumps(config))\n            manifest = {"scenarios": make_scenarios(t[50:74], config["protocol"]),\n                        "config_sha256": hashlib.sha256(config_path.read_bytes()).hexdigest(),\n                        "common_timestamps_float64_le_sha256": hashlib.sha256(t[50:74].astype(\'<f8\').tobytes()).hexdigest()}\n            (base / "diagnostic_clean_outages_manifest.json").write_text(json.dumps(manifest))\n            out = base / "diagnostic_output"\n            with patch("models.fusion_transformer.FusionTransformer", side_effect=lambda ablation: gps_only if ablation == "gps_only" else full), \\\n                    patch("models.fusion_lstm.FusionLSTM", return_value=lstm), \\\n                    patch("torch.load", side_effect=synthetic_state), \\\n                    patch("torch.cuda.is_available", return_value=False), contextlib.redirect_stdout(io.StringIO()):\n                run(config_path, r7_root=root, out_dir=out)\n            with (out / "summary.csv").open() as stream:\n                rows = list(csv.DictReader(stream))\n            self.assertEqual(len(rows), 60)\n            unique_rows(rows)\n            self.assertEqual(json.loads((out / "status.json").read_text())["status"], "complete")\n            for row in rows:\n                with np.load(out / row["prediction_file"], allow_pickle=False) as bundle:\n                    self.assertEqual(len(bundle["timestamps_s"]), 24)\n                    expected = raw_metrics(bundle["pred"], bundle["gt"], bundle["availability_mask"])\n                    self.assertAlmostEqual(float(row["rmse_3d_m"]), expected["rmse_3d_m"], places=12)\n                    np.testing.assert_array_equal(bundle["timestamps_s"], t[50:74])\n                    self.assertEqual(mask_sha256(bundle["availability_mask"]), row["mask_uint8_sha256"])\n            catalog = json.loads((out / "artifact_manifest.json").read_text())\n            for relative, info in catalog.items():\n                self.assertEqual(hashlib.sha256((out / relative).read_bytes()).hexdigest(), info["sha256"])\n            with (out / "paired_differences.csv").open() as stream:\n                pairs = list(csv.DictReader(stream))\n            self.assertEqual(len(pairs), 10)\n            for pair in pairs:\n                subset = {r["method"]: r for r in rows if r["scenario_id"] == pair["scenario_id"]}\n                delta = float(subset["fusion_full"]["rmse_3d_m"]) - float(subset["fusion_gps_only"]["rmse_3d_m"])\n                self.assertAlmostEqual(float(pair["full_minus_fusion_gps_only_rmse_3d_m"]), delta)\n\n    def test_existing_output_is_never_overwritten(self):\n        with tempfile.TemporaryDirectory() as temporary:\n            path = reserve_output(Path(temporary) / "diagnostic")\n            (path / "sentinel").write_text("unchanged")\n            with self.assertRaises(FileExistsError):\n                reserve_output(path)\n            self.assertEqual((path / "sentinel").read_text(), "unchanged")\n\n    def test_duplicate_evaluation_rows_rejected(self):\n        row = {"scenario_id": "a", "method": "b", "seed": 0}\n        with self.assertRaisesRegex(ValueError, "duplicate"):\n            unique_rows([row, row.copy()])\n\n    def test_artifact_identity_missing_and_tampered(self):\n        with tempfile.TemporaryDirectory() as temporary:\n            root = Path(temporary) / "mounted"\n            file = root / "drone/data/processed/t_imu.npy"\n            file.parent.mkdir(parents=True)\n            file.write_bytes(b"synthetic")\n            config = {"r7": {"artifacts": {"drone/data/processed/t_imu.npy": {\n                "bytes": 9, "sha256": hashlib.sha256(b"synthetic").hexdigest()}}}}\n            self.assertEqual(discover_r7(temporary, config)[0], root)\n            file.write_bytes(b"tampered!")\n            with self.assertRaisesRegex(ValueError, "identity mismatch"):\n                verify_r7(root, config)\n            file.unlink()\n            with self.assertRaisesRegex(FileNotFoundError, "Missing R7"):\n                verify_r7(root, config)\n\n    def test_synthetic_models_and_self_contained_bundle(self):\n        import torch\n        from models.fusion_transformer import FusionTransformer\n        from models.fusion_lstm import FusionLSTM\n        torch.manual_seed(0)\n        torch.set_num_threads(1)\n        n, window = 16, 8\n        t = np.arange(n, dtype=float)\n        imu = np.ones((n, 6), np.float32)\n        gps = np.repeat(t[:, None], 3, axis=1).astype(np.float32)\n        mask = availability_from_blocks(n, [(5, 11)])\n        clean = clean_gnss(t, gps, np.arange(n), t, mask)\n        tiny_models = [FusionTransformer(d_model=8, n_heads=2, n_layers=1, dropout=0),\n                       FusionTransformer(d_model=8, n_heads=2, n_layers=1, dropout=0, ablation="gps_only"),\n                       FusionLSTM(hidden=8, n_layers=1, dropout=0)]\n        for model in tiny_models:\n            for zero in (False, True):\n                pred = infer_windows(model, imu, clean["held_gnss"], mask, window, 1, "cpu", zero_imu=zero)\n                self.assertEqual(pred.shape, (n, 3))\n                self.assertTrue(np.isfinite(pred).all())\n                with torch.inference_mode():\n                    ref = model(torch.from_numpy(imu[:8] * (0 if zero else 1))[None],\n                                torch.from_numpy(clean["held_gnss"][:8])[None],\n                                torch.from_numpy(mask[:8].astype(np.float32))[None, :, None]).numpy()[0]\n                np.testing.assert_allclose(pred[:8], ref, rtol=1e-5, atol=1e-6)\n        with tempfile.TemporaryDirectory() as temporary:\n            data = {"gt": gps, "t": t, "original_indices": np.arange(n)}\n            scenario = {"scenario_id": "synthetic_test_only"}\n            metadata = {"checkpoint": None, "synthetic_random_model": True}\n            save_prediction(Path(temporary) / "bundle", pred, data, clean, scenario, metadata)\n            with np.load(Path(temporary) / "bundle/prediction.npz", allow_pickle=False) as bundle:\n                for key in ("pred", "gt", "timestamps_s", "availability_mask", "held_source_index",\n                            "error_3d_m", "correction_xyz_m", "scenario_id", "metadata_json"):\n                    self.assertIn(key, bundle.files)\n                np.testing.assert_array_equal(bundle["availability_mask"], mask)\n                np.testing.assert_allclose(bundle["error_3d_m"], raw_errors(pred, gps)["error_3d_m"])\n                self.assertEqual(json.loads(str(bundle["metadata_json"])), metadata)\n            csv_path = Path(temporary) / "summary.csv"\n            write_csv(csv_path, [{"scenario_id": "synthetic", **raw_metrics(pred, gps, mask)}])\n            self.assertEqual(len(csv_path.read_text().splitlines()), 2)\n            with self.assertRaises(FileExistsError):\n                write_csv(csv_path, [{"scenario_id": "duplicate"}])\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'configs/diagnostic_clean_outages.json': '{\n  "protocol_id": "r7_val_clean_outages_v1",\n  "description": "Locked validation diagnostic; no training, tuning or online-navigation claim.",\n  "r7": {\n    "kernel": "shapok/drone-nav-full-run",\n    "version": 7,\n    "output_id": 352644733,\n    "commit": "766575a07886e335c1d6cac664c5e140550942aa",\n    "url": "https://www.kaggle.com/code/shapok/drone-nav-full-run?scriptVersionId=352644733",\n    "artifacts": {\n      "drone/data/processed/t_imu.npy": {\n        "bytes": 216528,\n        "sha256": "6a97d036b4410ea133f9a61aad3f12f01b88d40f250d0fe10019a72a0a721df1"\n      },\n      "drone/data/processed/imu.npy": {\n        "bytes": 649328,\n        "sha256": "295b5a275fda83f64ff41265976a2076c7fdafb2de21b59131d8187ceae67855"\n      },\n      "drone/data/processed/t_gps.npy": {\n        "bytes": 649480,\n        "sha256": "d81130c811695894788cce9264d58f059f3cb0824f4867cfb25786095f8b9f63"\n      },\n      "drone/data/processed/gps_pos.npy": {\n        "bytes": 974156,\n        "sha256": "1db0dfb6108e590fdab6ad813eda642639e6d88a4ae6a576b45715a540649df7"\n      },\n      "drone/data/processed/t_gt.npy": {\n        "bytes": 21776,\n        "sha256": "db7063ed5333bd418de5c28fa155292b2966113dcf152610458b883303168562"\n      },\n      "drone/data/processed/gt_pos.npy": {\n        "bytes": 32600,\n        "sha256": "acb513c194a22ba9080e140506e325cfdc9446730439a70edf922e061af5ee6d"\n      },\n      "drone/data/processed/meta.json": {\n        "bytes": 343,\n        "sha256": "e498ca75f30575372f2b77a767486455aaa1c869a1e6ef54e67d4fa0ec221603"\n      },\n      "outputs/checkpoints/fusion_transformer_full_s0_outage0.0.pt": {\n        "bytes": 5332123,\n        "sha256": "8adacc2e7c16ed7d828ba4d49fab87dda10eeb908026b23f839d974a82a8140f"\n      },\n      "outputs/checkpoints/fusion_transformer_gps_only_s0_outage0.0.pt": {\n        "bytes": 5332375,\n        "sha256": "f2ad679aa5b797bf882e4e192ed910b1cf329795600960eeaae5ef1122bcd177"\n      },\n      "outputs/checkpoints/fusion_lstm_full_s0_outage0.0.pt": {\n        "bytes": 1780545,\n        "sha256": "e8b47d6109e521f5ddd938867dd91fa1086eb93f590deb7c714047ffe87383a5"\n      }\n    },\n    "model_source_sha256": {\n      "src/models/fusion_transformer.py": "c1c6172c1cc803640970c1a4b52963953db831634522fcc7c77eca8bb951e8e5",\n      "src/models/fusion_lstm.py": "a374b21d4601e9af9fad6186ae365a519e6132a613f25e9dbb800208a555171e"\n    }\n  },\n  "split": {\n    "name": "validation",\n    "train_fraction": 0.7,\n    "val_fraction": 0.1,\n    "expected_total_imu": 27050,\n    "expected_original_indices": [\n      18935,\n      21640\n    ],\n    "expected_common_indices": [\n      18935,\n      21623\n    ]\n  },\n  "window": 192,\n  "protocol": {\n    "durations_s": [\n      10,\n      30,\n      60\n    ],\n    "start_fractions": [\n      0.25,\n      0.5,\n      0.75\n    ],\n    "warmup_s": 30,\n    "recovery_s": 20,\n    "generator_seed": 0,\n    "rng_used": false,\n    "interval": "[onset, onset + duration)",\n    "start_rule": "fractions of [common_first + warmup, common_last - recovery - max_duration], snap forward to first timestamp"\n  },\n  "constant_velocity": {\n    "history_s": 5,\n    "min_span_s": 1,\n    "fit": "unweighted least squares per XYZ over unique accepted source timestamps",\n    "fallback": "zero velocity if fewer than two source timestamps or span < 1 s",\n    "anchor": "last accepted pre-onset position, extrapolation starts at its source timestamp"\n  },\n  "inference": {\n    "seed": 0,\n    "batch_windows": 1,\n    "device": "auto",\n    "allow_tf32": false,\n    "checkpoint_loading": "weights_only=True; strict=True",\n    "transformer_semantics": "noncausal window model; no added causal masks",\n    "window_state": "reset each window; no GT initialization"\n  },\n  "methods": [\n    {\n      "id": "held_gnss",\n      "kind": "baseline"\n    },\n    {\n      "id": "constant_velocity_gnss",\n      "kind": "baseline"\n    },\n    {\n      "id": "fusion_full",\n      "kind": "saved_model",\n      "architecture": "fusion_transformer",\n      "ablation": "full",\n      "seed": 0,\n      "checkpoint": "outputs/checkpoints/fusion_transformer_full_s0_outage0.0.pt"\n    },\n    {\n      "id": "fusion_gps_only",\n      "kind": "separately_trained_ablation",\n      "architecture": "fusion_transformer",\n      "ablation": "gps_only",\n      "seed": 0,\n      "checkpoint": "outputs/checkpoints/fusion_transformer_gps_only_s0_outage0.0.pt"\n    },\n    {\n      "id": "fusion_lstm",\n      "kind": "saved_model",\n      "architecture": "fusion_lstm",\n      "ablation": "full",\n      "seed": 0,\n      "checkpoint": "outputs/checkpoints/fusion_lstm_full_s0_outage0.0.pt"\n    },\n    {\n      "id": "fusion_full_zero_imu",\n      "kind": "fixed_checkpoint_input_sensitivity",\n      "architecture": "fusion_transformer",\n      "ablation": "full",\n      "seed": 0,\n      "checkpoint": "outputs/checkpoints/fusion_transformer_full_s0_outage0.0.pt",\n      "zero_imu": true\n    }\n  ],\n  "metrics": {\n    "alignment": "none",\n    "outage_scope": "mask=False samples only",\n    "control_scope": "entire common trace",\n    "averaging": "uniform per evaluated timestamp, not time-weighted",\n    "horizontal_axes": [\n      0,\n      1\n    ],\n    "vertical_axis": 2,\n    "units": "m in existing project coordinates; coordinate agreement unresolved",\n    "aggregation": "one row per method/scenario/seed; no seed or flight dispersion"\n  }\n}\n', 'configs/diagnostic_clean_outages_manifest.json': '{\n  "protocol_id": "r7_val_clean_outages_v1",\n  "status": "prepared_timestamp_only_no_R7_inference",\n  "r7_output_id": 352644733,\n  "config_sha256": "92e60cabcfc6ad31ddded0a33acd59dea49b953e6f9a9c4a6be3af1404cbc6c7",\n  "selection_inputs": "R7 validation timestamps and original GT support only; no error/position-based selection",\n  "common_indices_half_open": [\n    18935,\n    21623\n  ],\n  "common_timestamps_float64_le_sha256": "65f65ce416cd8b5a2b4d4e89fefb8b81cea2da9ecf5149d38e4c637eb5d1b110",\n  "mask_encoding": "unavailable_index_runs are half-open, relative to common grid; all other indices available; SHA256 of uint8 availability bytes",\n  "scenarios": [\n    {\n      "scenario_id": "val_control",\n      "split": "validation",\n      "episode_id": null,\n      "requested_duration_s": 0,\n      "source_blocked_intervals_s": [],\n      "unavailable_index_runs": [],\n      "n_unavailable": 0,\n      "metric_scope": "whole_common_trace",\n      "n_common": 2688,\n      "mask_uint8_sha256": "588b1bea0cec08685bbc845b232ccb924e129421942bd1ab3065b6e7f79a3bef",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e1_d10s",\n      "split": "validation",\n      "episode_id": 1,\n      "requested_duration_s": 10,\n      "target_onset_s": 1975.82550675,\n      "onset_s": 1975.869753,\n      "requested_cutoff_s": 1985.869753,\n      "last_unavailable_s": 1985.7717619999999,\n      "first_link_available_after_s": 1985.871752,\n      "observed_mask_interval_s": 10.001999000000069,\n      "unavailable_sample_span_s": 9.902008999999907,\n      "available_history_span_s": 70.11699599999997,\n      "recovery_span_s": 190.17200400000002,\n      "source_blocked_intervals_s": [\n        [\n          1975.869753,\n          1985.869753\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          686,\n          786\n        ]\n      ],\n      "n_unavailable": 100,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "9478793466ccff3f5a44e05df5b253ad2f7cc6695f5487567ebcd1da995cbd7d",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e1_d30s",\n      "split": "validation",\n      "episode_id": 1,\n      "requested_duration_s": 30,\n      "target_onset_s": 1975.82550675,\n      "onset_s": 1975.869753,\n      "requested_cutoff_s": 2005.869753,\n      "last_unavailable_s": 2005.7787529999998,\n      "first_link_available_after_s": 2005.878753,\n      "observed_mask_interval_s": 30.009000000000015,\n      "unavailable_sample_span_s": 29.908999999999878,\n      "available_history_span_s": 70.11699599999997,\n      "recovery_span_s": 170.16500300000007,\n      "source_blocked_intervals_s": [\n        [\n          1975.869753,\n          2005.869753\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          686,\n          986\n        ]\n      ],\n      "n_unavailable": 300,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "7d0af73a47f34df6d72982ada3badbd63efe284f363d489628995b4f3afd9881",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e1_d60s",\n      "split": "validation",\n      "episode_id": 1,\n      "requested_duration_s": 60,\n      "target_onset_s": 1975.82550675,\n      "onset_s": 1975.869753,\n      "requested_cutoff_s": 2035.869753,\n      "last_unavailable_s": 2035.788752,\n      "first_link_available_after_s": 2035.888753,\n      "observed_mask_interval_s": 60.019000000000005,\n      "unavailable_sample_span_s": 59.918998999999985,\n      "available_history_span_s": 70.11699599999997,\n      "recovery_span_s": 140.15500300000008,\n      "source_blocked_intervals_s": [\n        [\n          1975.869753,\n          2035.869753\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          686,\n          1286\n        ]\n      ],\n      "n_unavailable": 600,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "0b086cc043ee5ea7afba90e7fadd076edd9bd6b2a0d2f616dbb5d05f011e7724",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e2_d10s",\n      "split": "validation",\n      "episode_id": 2,\n      "requested_duration_s": 10,\n      "target_onset_s": 2015.8982565000001,\n      "onset_s": 2015.9817529999998,\n      "requested_cutoff_s": 2025.9817529999998,\n      "last_unavailable_s": 2025.8857529999998,\n      "first_link_available_after_s": 2025.986754,\n      "observed_mask_interval_s": 10.00500100000022,\n      "unavailable_sample_span_s": 9.903999999999996,\n      "available_history_span_s": 110.22899599999982,\n      "recovery_span_s": 150.057002,\n      "source_blocked_intervals_s": [\n        [\n          2015.9817529999998,\n          2025.9817529999998\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          1087,\n          1187\n        ]\n      ],\n      "n_unavailable": 100,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "f872ab438fa975ee018f3d6c65dd8caa39fc890e192c83c75c42d8209d0f0f0b",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e2_d30s",\n      "split": "validation",\n      "episode_id": 2,\n      "requested_duration_s": 30,\n      "target_onset_s": 2015.8982565000001,\n      "onset_s": 2015.9817529999998,\n      "requested_cutoff_s": 2045.9817529999998,\n      "last_unavailable_s": 2045.893754,\n      "first_link_available_after_s": 2045.993796,\n      "observed_mask_interval_s": 30.012043000000176,\n      "unavailable_sample_span_s": 29.912001000000146,\n      "available_history_span_s": 110.22899599999982,\n      "recovery_span_s": 130.04996000000006,\n      "source_blocked_intervals_s": [\n        [\n          2015.9817529999998,\n          2045.9817529999998\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          1087,\n          1387\n        ]\n      ],\n      "n_unavailable": 300,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "29d5854659ff5815929f2757a901e6ec74ef2f31a39ca9f9ced5a7f01b969755",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e2_d60s",\n      "split": "validation",\n      "episode_id": 2,\n      "requested_duration_s": 60,\n      "target_onset_s": 2015.8982565000001,\n      "onset_s": 2015.9817529999998,\n      "requested_cutoff_s": 2075.981753,\n      "last_unavailable_s": 2075.9067919999998,\n      "first_link_available_after_s": 2076.006753,\n      "observed_mask_interval_s": 60.02500000000032,\n      "unavailable_sample_span_s": 59.92503899999997,\n      "available_history_span_s": 110.22899599999982,\n      "recovery_span_s": 100.03700299999991,\n      "source_blocked_intervals_s": [\n        [\n          2015.9817529999998,\n          2075.981753\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          1087,\n          1687\n        ]\n      ],\n      "n_unavailable": 600,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "4f34e8d92654f4db91151dabf1fca1c766b9ecc507dab76928cbaee378224c20",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e3_d10s",\n      "split": "validation",\n      "episode_id": 3,\n      "requested_duration_s": 10,\n      "target_onset_s": 2055.97100625,\n      "onset_s": 2055.997753,\n      "requested_cutoff_s": 2065.997753,\n      "last_unavailable_s": 2065.903754,\n      "first_link_available_after_s": 2066.003758,\n      "observed_mask_interval_s": 10.006004999999732,\n      "unavailable_sample_span_s": 9.906000999999833,\n      "available_history_span_s": 150.24499600000013,\n      "recovery_span_s": 110.0399980000002,\n      "source_blocked_intervals_s": [\n        [\n          2055.997753,\n          2065.997753\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          1487,\n          1587\n        ]\n      ],\n      "n_unavailable": 100,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "0074eb8d8dc7bc8e4228c9480866b762e70819a1287206fa3700ff486e693957",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e3_d30s",\n      "split": "validation",\n      "episode_id": 3,\n      "requested_duration_s": 30,\n      "target_onset_s": 2055.97100625,\n      "onset_s": 2055.997753,\n      "requested_cutoff_s": 2085.997753,\n      "last_unavailable_s": 2085.910765,\n      "first_link_available_after_s": 2086.010757,\n      "observed_mask_interval_s": 30.01300399999991,\n      "unavailable_sample_span_s": 29.91301199999998,\n      "available_history_span_s": 150.24499600000013,\n      "recovery_span_s": 90.03299900000002,\n      "source_blocked_intervals_s": [\n        [\n          2055.997753,\n          2085.997753\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          1487,\n          1787\n        ]\n      ],\n      "n_unavailable": 300,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "c42dd6904e081422b95fc0d90f246f1dfddf40104196dc8094da0e0c6cb7178d",\n      "generator_seed": 0,\n      "rng_used": false\n    },\n    {\n      "scenario_id": "val_e3_d60s",\n      "split": "validation",\n      "episode_id": 3,\n      "requested_duration_s": 60,\n      "target_onset_s": 2055.97100625,\n      "onset_s": 2055.997753,\n      "requested_cutoff_s": 2115.997753,\n      "last_unavailable_s": 2115.9237519999997,\n      "first_link_available_after_s": 2116.023753,\n      "observed_mask_interval_s": 60.02599999999984,\n      "unavailable_sample_span_s": 59.92599899999959,\n      "available_history_span_s": 150.24499600000013,\n      "recovery_span_s": 60.02000300000009,\n      "source_blocked_intervals_s": [\n        [\n          2055.997753,\n          2115.997753\n        ]\n      ],\n      "unavailable_index_runs": [\n        [\n          1487,\n          2087\n        ]\n      ],\n      "n_unavailable": 600,\n      "metric_scope": "outage_only",\n      "n_common": 2688,\n      "mask_uint8_sha256": "a6662edb1addc52e5cd4647169edf53c5742e2106643354ecb051cd3a83ee997",\n      "generator_seed": 0,\n      "rng_used": false\n    }\n  ]\n}\n'}
PACKAGED_SHA256 = {'src/diagnostics/__init__.py': 'fd3a6eb7a18cd1cc76c1ac952d11562fc8a6ca0744f9dae7fc405954e09f3894', 'src/diagnostics/clean_outages.py': 'f86dfb712be86f6c1a2c03d18d77cb4800339337fdd57924d7cee8d5ec9b80b4', 'src/diagnostics/run_clean_outages.py': '2ed5ffc727d8deabf5931cc19bcdfb9ad1380a8cdfa70135cd8d0d236275f36b', 'src/models/fusion_transformer.py': 'c1c6172c1cc803640970c1a4b52963953db831634522fcc7c77eca8bb951e8e5', 'src/models/fusion_lstm.py': 'a374b21d4601e9af9fad6186ae365a519e6132a613f25e9dbb800208a555171e', 'tests/test_clean_outages.py': '41931a41b44d1e75157051d2c80aaa50306fa3b784b7000451164f35e699f47b', 'configs/diagnostic_clean_outages.json': '92e60cabcfc6ad31ddded0a33acd59dea49b953e6f9a9c4a6be3af1404cbc6c7', 'configs/diagnostic_clean_outages_manifest.json': 'a12adbe63ee0c2d57a8d2ac0fab2f7895f584abc35a240440498140c42681ca4'}

for relative, source in PACKAGED_FILES.items():
    payload = source.encode('utf-8')
    assert hashlib.sha256(payload).hexdigest() == PACKAGED_SHA256[relative]
    destination = CODE_ROOT / relative
    if destination.exists():
        if destination.read_bytes() != payload:
            raise RuntimeError(f'Existing diagnostic code differs: {destination}; use a fresh session')
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(payload)
sys.path.insert(0, str(CODE_ROOT / 'src'))
CONFIG_PATH = CODE_ROOT / 'configs/diagnostic_clean_outages.json'
print(f'Packaged {len(PACKAGED_FILES)} verified source/config files into {CODE_ROOT}')


## Проверки до inference

Unit/smoke tests используют только маленькие синтетические данные и случайные уменьшенные
модели. Их численные значения не являются экспериментальными результатами R7.
Следующая ячейка сверяет настоящий input, общий validation-диапазон и заранее
зафиксированные сценарии; при несовпадении запуск останавливается.


In [ ]:
import subprocess
subprocess.run([sys.executable, '-B', '-m', 'unittest', 'discover',
                '-s', str(CODE_ROOT / 'tests'), '-p', 'test_clean_outages.py', '-v'],
               cwd=CODE_ROOT, check=True)


In [ ]:
import json
import numpy as np
from diagnostics.run_clean_outages import discover_r7, verify_r7, prepare_validation
from diagnostics.clean_outages import make_scenarios

config = json.loads(CONFIG_PATH.read_text())
if R7_ROOT is None:
    R7_ROOT, verified = discover_r7(INPUT_ROOT, config)
else:
    R7_ROOT = Path(R7_ROOT)
    verified = verify_r7(R7_ROOT, config)
prepared = prepare_validation(R7_ROOT, config)
scenarios = make_scenarios(prepared['t'], config['protocol'])
locked = json.loads((CONFIG_PATH.parent / 'diagnostic_clean_outages_manifest.json').read_text())
assert scenarios == locked['scenarios'], 'Locked scenarios changed'
assert hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest() == locked['config_sha256']
assert hashlib.sha256(prepared['t'].astype('<f8').tobytes()).hexdigest() == locked['common_timestamps_float64_le_sha256']
print('Verified R7 input:', R7_ROOT)
print(json.dumps(prepared['support'], indent=2))
for scenario in scenarios:
    print(scenario['scenario_id'], 'onset=', scenario.get('onset_s'),
          'cutoff=', scenario.get('requested_cutoff_s'),
          'unavailable points=', scenario['n_unavailable'])
assert not OUTPUT_DIR.exists(), f'Results directory exists: {OUTPUT_DIR}; use a fresh session'


## Оценка сохранённых моделей

1 контроль + 9 отдельных эпизодов (3 начала × 10/30/60 секунд), 6 строк на сценарий.
Основная ошибка — raw RMSE на недоступных отсчётах, **без выравнивания**. Контроль использует
весь общий validation-отрезок. Первые две существующие оси считаются горизонтальными,
третья — вертикальной; согласование координат R7 ещё требует отдельной проверки.
Протокол и baseline скорости зафиксированы до просмотра ошибок. Результат без преимущества
full также допустим. Один seed и эпизоды одного полёта не дают межseed/межполётной статистики.


In [ ]:
from diagnostics.run_clean_outages import run
result_dir = run(CONFIG_PATH, input_root=INPUT_ROOT, r7_root=R7_ROOT, out_dir=OUTPUT_DIR)
print('Diagnostic artifacts:', result_dir)


In [ ]:
import csv
from IPython.display import Markdown, display
status = json.loads((OUTPUT_DIR / 'status.json').read_text())
assert status['status'] == 'complete' and status['evaluation_rows'] == 60
with (OUTPUT_DIR / 'paired_differences.csv').open() as stream:
    pairs = list(csv.DictReader(stream))
header = '| Scenario | full − gps_only, m | full − held, m | full − CV, m | zero IMU − full, m |'
table = [header, '|---|---:|---:|---:|---:|']
for row in pairs:
    values = [row[key] for key in ('full_minus_fusion_gps_only_rmse_3d_m',
              'full_minus_held_gnss_rmse_3d_m', 'full_minus_constant_velocity_gnss_rmse_3d_m',
              'sensitivity_zero_imu_minus_full_rmse_3d_m')]
    table.append('| ' + row['scenario_id'] + ' | ' + ' | '.join(f'{float(v):.6f}' for v in values) + ' |')
display(Markdown('\n'.join(table)))
print(json.dumps(status, indent=2))
print('For analysis, download outputs/diagnostic_clean_outages/ in full, including prediction.npz bundles.')


## Что передать для анализа

Скачайте весь `outputs/diagnostic_clean_outages/` и output/log этой версии notebook.
В каталоге будут `summary.csv` (60 уникальных evaluation-строк),
`paired_differences.csv` (10 сценариев), manifest/config/provenance/status,
snapshot исполненного кода и файлы каждого сценария с масками и происхождением GNSS.
Каждый из 60 `prediction.npz` содержит pred, GT, timestamps, scenario ID,
метаданные checkpoint, ошибки по всей временной линии и `prediction − held-GNSS`.
NPZ читается с `allow_pickle=False`. Положительные/отрицательные разности в таблице —
разности ошибок, а не доверительные интервалы. Zero-IMU — чувствительность фиксированного
checkpoint; gps_only — другой, ранее обученный checkpoint.

Исходный `results.csv`, checkpoints и notebook R7 не изменяются. Основной вопрос анализа:
есть ли преимущество full над отдельно обученной gps_only, удержанием координат
и причинной экстраполяцией при действительно закрытых измерениях?
